<a href="https://colab.research.google.com/github/selin8803/Project-in-Machine-Learning-Accelerating-KMeans/blob/main/Stage1_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 1 – K-Means Baseline and GPU Implementation

## 1. Environment Setup

This notebook implements K-Means from scratch on CPU and GPU,
using Fashion-MNIST as the dataset.

The goal is to compare correctness, runtime, and clustering quality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import subprocess

print("NumPy version:", np.__version__)

# Check GPU
!nvidia-smi

# Check CUDA compiler
!nvcc --version

NumPy version: 2.4.6
Sun Nov  8 11:52:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------

In [ ]:
import torch
from torchvision import datasets

# Load Fashion-MNIST
train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True
)

# Convert images to NumPy arrays
X_train = train_dataset.data.numpy().astype(np.float32)
X_test = test_dataset.data.numpy().astype(np.float32)

# Flatten images: 28x28 -> 784
X_train = X_train.reshape(-1, 784)
X_test = X_test.reshape(-1, 784)

# Normalize pixel values to [0, 1]
X_train /= 255.0
X_test /= 255.0

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Minimum value:", X_train.min())
print("Maximum value:", X_train.max())

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 213kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.93MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 4.25MB/s]

Training shape: (60000, 784)
Test shape: (10000, 784)
Minimum value: 0.0
Maximum value: 1.0


## 2. CPU K-Means Implementation

In this section, we implement Lloyd's K-Means algorithm
from scratch using NumPy.

The implementation includes centroid initialization,
cluster assignment, centroid updating, and convergence checking.

In [ ]:
# K-Means parameters
K = 10
MAX_ITER = 100
TOL = 1e-4

# Initialize centroids randomly
def initialize_centroids(X, K, seed=42):
    np.random.seed(seed)

    indices = np.random.choice(len(X), K, replace=False)

    return X[indices].copy()

### 2.1 Cluster Assignment

For each data point, calculate its squared Euclidean
distance to all centroids and assign it to the nearest one.

In [ ]:
def assign_clusters(X, centroids):

    # Calculate squared distances efficiently
    X_norm = np.sum(X ** 2, axis=1, keepdims=True)
    C_norm = np.sum(centroids ** 2, axis=1)

    distances = X_norm + C_norm - 2 * X @ centroids.T

    # Avoid small negative values caused by floating-point errors
    distances = np.maximum(distances, 0)

    # Assign each point to its nearest centroid
    labels = np.argmin(distances, axis=1)

    return labels

In [ ]:
# Initialize centroids
centroids = initialize_centroids(X_train, K)

# Assign points to clusters
labels = assign_clusters(X_train, centroids)

print("Centroids shape:", centroids.shape)
print("Labels shape:", labels.shape)
print("Number of clusters:", len(np.unique(labels)))

Centroids shape: (10, 784)
Labels shape: (60000,)
Number of clusters: 10


### 2.2 Centroid Update

Update each centroid by calculating the mean
of all data points assigned to its cluster.

Empty clusters retain their previous centroid.

In [ ]:
def update_centroids(X, labels, old_centroids, K):

    new_centroids = np.zeros_like(old_centroids)

    # Count points in each cluster
    counts = np.bincount(labels, minlength=K)

    # Sum points belonging to each cluster
    np.add.at(new_centroids, labels, X)

    # Calculate the mean for non-empty clusters
    for k in range(K):
        if counts[k] > 0:
            new_centroids[k] /= counts[k]
        else:
            # Keep the previous centroid if the cluster is empty
            new_centroids[k] = old_centroids[k]

    return new_centroids

In [ ]:
new_centroids = update_centroids(
    X_train,
    labels,
    centroids,
    K
)

print("New centroids shape:", new_centroids.shape)

print("Centroids updated:",
      not np.allclose(centroids, new_centroids))

New centroids shape: (10, 784)
Centroids updated: True


### 2.3 Complete K-Means Algorithm

Combine cluster assignment and centroid updating
into an iterative Lloyd's algorithm.

The algorithm stops when centroid movement
is smaller than the convergence tolerance.

In [ ]:
def kmeans(X, K, max_iter=100, tol=1e-4, seed=42):

    # Step 1: Initialize centroids
    centroids = initialize_centroids(X, K, seed)

    for iteration in range(max_iter):

        # Step 2: Assign points to nearest centroids
        labels = assign_clusters(X, centroids)

        # Step 3: Update centroids
        new_centroids = update_centroids(
            X, labels, centroids, K
        )

        # Step 4: Calculate centroid movement
        shift = np.linalg.norm(new_centroids - centroids)

        centroids = new_centroids

        # Step 5: Check convergence
        if shift < tol:
            break

    # Recompute assignments for the final centroids
    labels = assign_clusters(X, centroids)

    # Calculate final inertia
    inertia = np.sum(
        (X - centroids[labels]) ** 2
    )

    return centroids, labels, inertia, iteration + 1

In [ ]:
# Use a subset for the initial test
X_small = X_train[:5000]

start_time = time.perf_counter()

centroids_cpu, labels_cpu, inertia_cpu, n_iter = kmeans(
    X_small,
    K=10,
    max_iter=100,
    tol=1e-4,
    seed=42
)

cpu_time = time.perf_counter() - start_time

print("CPU K-Means Results")
print("-------------------")
print("Centroids shape:", centroids_cpu.shape)
print("Labels shape:", labels_cpu.shape)
print("Number of iterations:", n_iter)
print("Inertia:", inertia_cpu)
print("Runtime:", cpu_time, "seconds")
print("Number of clusters:", len(np.unique(labels_cpu)))

CPU K-Means Results
-------------------
Centroids shape: (10, 784)
Labels shape: (5000,)
Number of iterations: 22
Inertia: 160083.66
Runtime: 2.825813961999984 seconds
Number of clusters: 10


### 2.4 Correctness Tests

We verify the CPU implementation using small numerical examples,
empty clusters, and different initializations.

In [ ]:
# Small manually constructed dataset
X_test_small = np.array([
    [0, 0],
    [2, 0],
    [10, 0],
    [12, 0]
], dtype=np.float32)

initial_centroids = np.array([
    [0, 0],
    [10, 0]
], dtype=np.float32)

# Assignment test
test_labels = assign_clusters(
    X_test_small,
    initial_centroids
)

print("Expected labels: [0 0 1 1]")
print("Actual labels:", test_labels)

assert np.array_equal(test_labels, [0, 0, 1, 1])

# Centroid update test
updated = update_centroids(
    X_test_small,
    test_labels,
    initial_centroids,
    K=2
)

expected_centroids = np.array([
    [1, 0],
    [11, 0]
], dtype=np.float32)

print("Updated centroids:")
print(updated)

assert np.allclose(updated, expected_centroids)

print("Small numerical test passed!")

Expected labels: [0 0 1 1]
Actual labels: [0 0 1 1]
Updated centroids:
[[ 1.  0.]
 [11.  0.]]
Small numerical test passed!


In [ ]:
X_empty = np.array([
    [0, 0],
    [2, 0],
    [10, 0],
    [12, 0]
], dtype=np.float32)

old_centroids = np.array([
    [0, 0],
    [10, 0],
    [99, 99]
], dtype=np.float32)

labels_empty = np.array([0, 0, 1, 1])

updated_empty = update_centroids(
    X_empty,
    labels_empty,
    old_centroids,
    K=3
)

print("Updated centroids:")
print(updated_empty)

assert np.allclose(updated_empty[2], old_centroids[2])

print("Empty cluster test passed!")

Updated centroids:
[[ 1.  0.]
 [11.  0.]
 [99. 99.]]
Empty cluster test passed!


In [ ]:
# First initialization
centroids_1, labels_1, inertia_1, iter_1 = kmeans(
    X_small, K=10, seed=42
)

# Second initialization
centroids_2, labels_2, inertia_2, iter_2 = kmeans(
    X_small, K=10, seed=123
)

print("Initialization 1:")
print("Inertia:", inertia_1)
print("Iterations:", iter_1)

print("\nInitialization 2:")
print("Inertia:", inertia_2)
print("Iterations:", iter_2)

assert not np.array_equal(
    initialize_centroids(X_small, 10, seed=42),
    initialize_centroids(X_small, 10, seed=123)
)

print("\nDifferent initialization test passed!")

Initialization 1:
Inertia: 160083.66
Iterations: 22

Initialization 2:
Inertia: 170498.6
Iterations: 18

Different initialization test passed!
